 
# 文本生成案例——莎士比亚文集

## 数据集加载

In [3]:
import tensorflow as tf
import os

# 获取当前工作目录下的 data 文件夹绝对路径
current_dir = os.getcwd()
target_dir = os.path.join(current_dir, 'data')

# get_file 的逻辑是：存储路径 = cache_dir / cache_subdir / filename
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt', 
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt',
    cache_dir=current_dir,    # 设置缓存根目录为当前目录
    cache_subdir='data'       # 设置子目录为 data
)

print(f"文件已保存至: {path_to_file}")

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
文件已保存至: d:\rozen_file\WangDao_PythonAI\Day17\data\shakespeare.txt


In [ ]:
with open("./data/shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# 构建字符级字典

# set(text): 将字符串 text 转换为一个集合（Set）。集合的特性是去重，所以这一步会去掉所有重复出现的字符，只保留唯一的字符
# list(...): 将集合转换回列表（List）
# sorted(...): 对列表进行排序
chars = sorted(list(set(text)))

# enumerate(chars):  返回enumerate(枚举)对象 (i, ch) 是 chars 中的每个字符及其索引
char2idx = {ch: i for i, ch in enumerate(chars)}  # 字符 -> 索引
idx2char = {i: ch for i, ch in enumerate(chars)}  # 索引 -> 字符

print(f"总共有 {len(chars)} 个不同的字符")

总共有 65 个不同的字符


In [10]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class TextDataset(Dataset):
    def __init__(self, text_ids, seq_len=101):
        self.text_ids = text_ids
        self.seq_len = seq_len
        self.num_seq = len(text_ids) // self.seq_len #样本的个数

    def __len__(self):
        return self.num_seq

    def __getitem__(self, idx):
        # 获取第idx个样本的输入和目标序列
        return self.text_ids[idx*self.seq_len:(idx+1)*self.seq_len]


text_ids = [char2idx[ch] for ch in text]  # 将文本转换为索引序列
dataset = TextDataset(text_ids, seq_len=101)  # 创建数据集


def collate_fn(batch):
    # batch shape: (batch_size, seq_len)
    batch = torch.tensor(batch, dtype=torch.long)
    # 前100个字符作为输入
    x = batch[:, :100]
    # 后100个字符作为label
    y = batch[:, 1:101]
    return x, y

dataloader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)


In [12]:
for x, y in dataloader:
    print(x.shape, y.shape)
    break

torch.Size([64, 100]) torch.Size([64, 100])


## 搭建模型

In [14]:
import torch.nn as nn

class CharRNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=1024, hidden_size=256, num_layers=1):
        super(CharRNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, hidden=None):
        # x: (batch_size, seq_len)
        x = self.embedding(x)  # (batch_size, seq_len, embed_dim)
        out, hidden = self.rnn(x, hidden)  # out: (batch_size, seq_len, hidden_size)
        out = self.fc(out)  # (batch_size, seq_len, vocab_size)
        return out, hidden

# 假设vocab_size已知
vocab_size = len(char2idx)
model = CharRNNModel(vocab_size)


In [15]:
# 简单的前向计算示例
# 假设有一个batch的数据x
x, y = next(iter(dataloader))
output, hidden = model(x)
print("output shape:", output.shape)  # 应为 (batch_size, seq_len, vocab_size)


output shape: torch.Size([64, 100, 65])
